# BASE LIGHTGBM SOLUTION

This notebook contains the optimized LightGBM model setup for generating competition submissions using **Polars** for optimized performance

**Best Model:** LightGBM Float32 with clipping
- **Score:** 0.217232 (best among all models)
- **Training time:** 53.97 seconds
- **Data:** Processed with clipping (0.003-0.997 quantiles)

**Purpose:**
Train LightGBM model and generate submission files for Kaggle competition

**Polars Benefits:**
- Faster data loading and processing
- Memory efficient operations
- Lazy evaluation capabilities
- Better multi-threading performance

## 1.1 IMPORTS AND CONFIGURATION

Setting up the environment with all necessary libraries and Polars configuration for optimal performance.

In [1]:
# ============================================
# IMPORTS AND CONFIGURATION
# ============================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import polars as pl
from pathlib import Path
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import cohen_kappa_score
import time
from collections import Counter
import lightgbm as lgb

# Polars configuration
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

# Set up paths
base_dir = Path("..")
full_train_path = base_dir / "data" / "processed" / "train_processed.parquet"
full_test_path = base_dir / "data" / "test.parquet"  # Original test data for submission
results_dir = base_dir / "Results"

print("✅ Imports and configuration complete")
print(f"📁 Results directory: {results_dir}")

✅ Imports and configuration complete
📁 Results directory: ..\Results


## 1.2 DATA LOADING

Loading the training dataset with clipping and test dataset for submission.

In [2]:
# ============================================
# LOAD DATASETS
# ============================================
print("="*60)
print("LOADING DATASETS")
print("="*60)

# Load training dataset (with clipping)
start_time = time.time()
train_full = pl.read_parquet(full_train_path)
load_time = time.time() - start_time

print(f"Train full: {train_full.shape}")
print(f"Load time: {load_time:.2f} seconds")

# Load test dataset (original for submission)
test_full = pl.read_parquet(full_test_path)
print(f"Test full: {test_full.shape}")

print(f"\n✅ Datasets loaded")
print(f"   Train: {train_full.shape}")
print(f"   Test: {test_full.shape}")

LOADING DATASETS
Train full: (5337414, 94)
Load time: 3.25 seconds
Test full: (1447107, 92)

✅ Datasets loaded
   Train: (5337414, 94)
   Test: (1447107, 92)


## 2.0 LIGHTGBM MODEL TRAINING

Training the best performing LightGBM model with Float32 data and clipping.

In [3]:
# ============================================
# LIGHTGBM MODEL TRAINING
# ============================================

print("="*60)
print("TRAINING LIGHTGBM MODEL")
print("="*60)

# Metryka
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

# Feature selection
feature_cols = [col for col in train_full.columns if col.startswith('feature_')]
X = train_full.select(feature_cols).fill_null(0).to_numpy()
y = train_full.select("y_target").to_numpy().ravel()
w = train_full.select("weight").to_numpy().ravel()

print(f"Features: {len(feature_cols)}")
print(f"X shape: {X.shape}")
print(f"X dtype: {X.dtype}")

# LightGBM model
start_time = time.time()
lgb_model = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',
    num_leaves=31,
    learning_rate=0.1,
    n_estimators=100,
    random_state=42,
    verbose=-1
)

lgb_model.fit(X, y, sample_weight=w)
lgb_time = time.time() - start_time

# Predykcja i score
y_pred_lgb = lgb_model.predict(X)
lgb_score = weighted_rmse_score(y, y_pred_lgb, w)

print(f"LightGBM training time: {lgb_time:.2f} seconds")
print(f"LightGBM Score: {lgb_score:.6f}")
print(f"\n🎯 Best model ready!")

TRAINING LIGHTGBM MODEL
Features: 86
X shape: (5337414, 86)
X dtype: float32
LightGBM training time: 53.97 seconds
LightGBM Score: 0.217232

🎯 Best model ready!


## 3.0 GENERATE SUBMISSION

Generate predictions on test data and create submission file for Kaggle.

In [4]:
# ============================================
# GENERATE SUBMISSION
# ============================================

print("="*60)
print("GENERATING SUBMISSION")
print("="*60)

# Prepare test features
X_test = test_full.select(feature_cols).fill_null(0).to_numpy()
print(f"Test features shape: {X_test.shape}")
print(f"Test features dtype: {X_test.dtype}")

# Generate predictions
start_time = time.time()
test_predictions = lgb_model.predict(X_test)
pred_time = time.time() - start_time

print(f"Prediction time: {pred_time:.2f} seconds")
print(f"Predictions shape: {test_predictions.shape}")
print(f"Predictions range: [{test_predictions.min():.6f}, {test_predictions.max():.6f}]")

# Create submission dataframe
submission_df = test_full.select(['id']).with_columns(
    pl.Series(name="prediction", values=test_predictions)
)

print(f"\nSubmission shape: {submission_df.shape}")
print("Sample predictions:")
print(submission_df.head())

GENERATING SUBMISSION
Test features shape: (1447107, 86)
Test features dtype: float64
Prediction time: 2.34 seconds
Predictions shape: (1447107,)
Predictions range: [-0.015234, 0.004567]

Submission shape: (1447107, 2)
Sample predictions:
shape: (5, 2)
┌──────────────────────────────────────┬─────────────┐
│ id                                    ┆ prediction  │
│ ---                                  ┆ ---         │
│ str                                  ┆ f64         │
╞════════════════════════════════════════╪═════════════╡
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__3_3647_95 ┆ -0.013161   │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__10_3647_88 ┆ -0.013091   │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__25_3647_71 ┆ -0.013138   │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__1_3647_97  ┆ -0.013245   │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__10_3648_87 ┆ -0.013216   │
└──────────────────────────────────────┴─────────────┘


## 4.0 SAVE SUBMISSION FILE

Save the submission file in the Results directory.

In [5]:
# ============================================
# SAVE SUBMISSION FILE
# ============================================

print("="*60)
print("SAVING SUBMISSION FILE")
print("="*60)

# Generate timestamp for filename
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission_filename = f"lightgbm_solution_{timestamp}.csv"
submission_path = results_dir / submission_filename

# Save submission
submission_df.write_csv(submission_path)

print(f"✅ Submission saved: {submission_path}")
print(f"📊 File size: {submission_path.stat().st_size / 1024**2:.2f} MB")
print(f"📝 Records: {len(submission_df)}")

# Verify file exists and show sample
if submission_path.exists():
    print("\n📋 File verification:")
    verify_df = pl.read_csv(submission_path)
    print(f"Loaded shape: {verify_df.shape}")
    print("Sample:")
    print(verify_df.head())
else:
    print("❌ File not found!")

SAVING SUBMISSION FILE
✅ Submission saved: ..\Results\lightgbm_solution_20260318_024956.csv
📊 File size: 35.67 MB
📝 Records: 1447107

📋 File verification:
Loaded shape: (1447107, 2)
Sample:
shape: (5, 2)
┌──────────────────────────────────────┬─────────────┐
│ id                                    ┆ prediction  │
│ ---                                  ┆ ---         │
│ str                                  ┆ f64         │
╞════════════════════════════════════════╪═════════════╡
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__3_3647_95 ┆ -0.013161   │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__10_3647_88 ┆ -0.013091   │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__25_3647_71 ┆ -0.013138   │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__1_3647_97  ┆ -0.013245   │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__10_3648_87 ┆ -0.013216   │
└──────────────────────────────────────┴─────────────┘


## 5.0 SUMMARY

Model training and submission generation complete!

In [6]:
# ============================================
# SUMMARY
# ============================================

print("="*60)
print("LIGHTGBM SOLUTION SUMMARY")
print("="*60)

print(f"🎯 Model: LightGBM Float32")
print(f"📊 Training Score: {lgb_score:.6f}")
print(f"⏱️ Training Time: {lgb_time:.2f} seconds")
print(f"📁 Data: Train with clipping (0.003-0.997)")
print(f"🔢 Features: {len(feature_cols)}")
print(f"📏 Train size: {train_full.shape}")
print(f"📏 Test size: {test_full.shape}")
print(f"💾 Submission: {submission_path}")
print(f"📊 Submission size: {submission_path.stat().st_size / 1024**2:.2f} MB")

print(f"\n🚀 Ready for Kaggle submission!")
print(f"\n📈 Model Performance:")
print(f"   - LightGBM: {lgb_score:.6f} ⭐ BEST")
print(f"   - XGBoost: 0.179537")
print(f"   - Linear Regression: 0.046329")

print(f"\n✨ LightGBM is the clear winner!")

LIGHTGBM SOLUTION SUMMARY
🎯 Model: LightGBM Float32
📊 Training Score: 0.217232
⏱️ Training Time: 53.97 seconds
📁 Data: Train with clipping (0.003-0.997)
🔢 Features: 86
📏 Train size: (5337414, 94)
📏 Test size: (1447107, 92)
💾 Submission: ..\Results\lightgbm_solution_20260318_024956.csv
📊 Submission size: 35.67 MB

🚀 Ready for Kaggle submission!

📈 Model Performance:
   - LightGBM: 0.217232 ⭐ BEST
   - XGBoost: 0.179537
   - Linear Regression: 0.046329

✨ LightGBM is the clear winner!
